In [16]:
import pandas as pd
import os

In [17]:
import os
import re

# Folder where your files are stored
folder = '/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/02_ET_data'

# Regex pattern:
# ^\d+_  → matches digits at the beginning followed by underscore
pattern = re.compile(r"^\d+_")

for filename in os.listdir(folder):
    old_path = os.path.join(folder, filename)

    # Skip folders
    if not os.path.isfile(old_path):
        continue

    # Apply renaming only if pattern matches
    if pattern.match(filename):
        new_filename = pattern.sub("", filename)  # Remove leading digits + underscore
        new_path = os.path.join(folder, new_filename)
        
        print(f"Renaming: {filename} → {new_filename}")
        os.rename(old_path, new_path)

print("Done!")


Done!


In [18]:
import pandas as pd
import os

def process_eye_blink_fullsegment(file_path, output_path):
    print(f"Processing file: {file_path}...")

    # Detect header row
    header_row_index = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if line.startswith("Row,Timestamp"):
                header_row_index = i
                break
    
    df = pd.read_csv(file_path, skiprows=header_row_index)
    df.columns = df.columns.str.strip()

    # Ensure Blink Detected exists
    if "Blink Detected" not in df.columns:
        print("Error: 'Blink Detected' column not found.")
        print("Available columns:", df.columns.tolist())
        return

    df = df[['Timestamp', 'SlideEvent', 'SourceStimuliName', 'Blink Detected']].copy()
    df['Blink Detected'] = pd.to_numeric(df['Blink Detected'], errors='coerce').fillna(0)
    
    # Start and End detection
    start_indices = df.index[df['SlideEvent'] == 'StartMedia'].tolist()
    end_indices = df.index[df['SlideEvent'] == 'EndMedia'].tolist()

    if not start_indices or not end_indices:
        print("Error: Could not find StartMedia / EndMedia.")
        return

    start_idx = start_indices[0]
    valid_end_indices = [i for i in end_indices if i > start_idx]
    if not valid_end_indices:
        print("Error: No EndMedia found AFTER StartMedia.")
        return

    end_idx = valid_end_indices[0]

    # Segment
    segment_df = df.loc[start_idx:end_idx].copy()

    # Time features
    start_time = segment_df['Timestamp'].iloc[0]
    segment_df['TimeElapsed'] = segment_df['Timestamp'] - start_time
    segment_df['Start_Index'] = start_idx
    segment_df['End_Index'] = end_idx
    segment_df['Start_Timestamp'] = start_time
    segment_df['End_Timestamp'] = segment_df['Timestamp'].iloc[-1]

    # Ensure folder exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Save output
    segment_df.to_csv(output_path, index=False)

    print(f"✔ Full blink segment saved to: {output_path}")
    print(f"Rows exported: {len(segment_df)}")
    print(f"Start index: {start_idx} | End index: {end_idx}")

In [21]:
import os
import glob

data_folder = '/Users/hjcho/Documents/GitHub/PC_HJ/data/00_raw_data/02_ET_data'
result_folder = '/Users/hjcho/Documents/GitHub/PC_HJ/data/01_processed_data/02_processed_ET'

os.makedirs(result_folder, exist_ok=True)

for filename in os.listdir(raw_folder):
    file_path = os.path.join(raw_folder, filename)

    # 🔥 MUST SKIP DIRECTORIES FIRST
    if os.path.isdir(file_path):
        print(f"Skipping directory: {filename}")
        continue

    # 🔥 Only process CSV files
    if not filename.lower().endswith(".csv"):
        print(f"Skipping non-CSV file: {filename}")
        continue

    # OUTPUT FILE NAME
    output_name = filename.replace(".csv", "_processed.csv")
    output_path = os.path.join(result_folder, output_name)

    print(f"\n=== Running {filename} ===")
    process_gsr_data_fullsegment(file_path, output_path)

NameError: name 'raw_folder' is not defined